# 03 – Fault Tree Analysis: a logikai modelltől a minimal cut setig

**3. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- AND/OR kapuk valószínűségi értelmezése
- fault tree reprezentálása Pythonban
- rekurzív kiértékelő felépítése
- minimal cut set és Birnbaum importance értelmezése

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. FTA gondolkodás

A kérdés: **mely alap-események vagy eseménykombinációk elegendők a top event bekövetkezéséhez?**

Független bemenetek esetén: AND → $\prod p_i$, OR → $1-\prod(1-p_i)$. A függetlenség itt explicit modellfeltevés.


In [ ]:
def AND(*p): return float(np.prod(p))
def OR(*p): return 1-float(np.prod([1-x for x in p]))
p_lshh,p_xv=.01,.005
print("AND",AND(p_lshh,p_xv)); print("OR",OR(p_lshh,p_xv))


## 2. Első explicit fault tree

Először nem használunk kész libraryt. A teljes logika látható `dict` struktúrában.


In [ ]:
tree={"top_event":"AUTO_FAIL","nodes":{
 "AUTO_FAIL":{"type":"OR","children":["LSHH_FOD","XV_FOD"]},
 "LSHH_FOD":{"type":"BASIC","probability":.01},
 "XV_FOD":{"type":"BASIC","probability":.005}}}
tree


## 3. Rekurzív evaluator

BASIC esetén probability-t adunk vissza; gate esetén előbb a gyermekeket értékeljük ki.


In [ ]:
def evaluate(tree,node_id=None):
    node_id=node_id or tree["top_event"]
    node=tree["nodes"][node_id]
    typ=node["type"]
    if typ=="BASIC": return float(node["probability"])
    ps=[evaluate(tree,c) for c in node["children"]]
    if typ=="AND": return float(np.prod(ps))
    if typ=="OR": return 1-float(np.prod([1-p for p in ps]))
    raise ValueError(typ)
print(evaluate(tree))


## 4. TK-101 overflow tree

Oktatási modell:

`OVERFLOW = HIGH_LEVEL_CHALLENGE AND AUTO_FAIL AND MANUAL_FAIL`

`AUTO_FAIL = LSHH_FOD OR XV_FOD`

`MANUAL_FAIL = LT101_LOW OR OPERATOR_MISS`


In [ ]:
tk={"top_event":"OVERFLOW","nodes":{
 "OVERFLOW":{"type":"AND","children":["CHALLENGE","AUTO_FAIL","MANUAL_FAIL"]},
 "AUTO_FAIL":{"type":"OR","children":["LSHH_FOD","XV_FOD"]},
 "MANUAL_FAIL":{"type":"OR","children":["LT101_LOW","OPERATOR_MISS"]},
 "CHALLENGE":{"type":"BASIC","probability":.08},
 "LSHH_FOD":{"type":"BASIC","probability":.01},
 "XV_FOD":{"type":"BASIC","probability":.005},
 "LT101_LOW":{"type":"BASIC","probability":.02},
 "OPERATOR_MISS":{"type":"BASIC","probability":.04}}}
print(f"P(OVERFLOW)={evaluate(tk):.8f}")


## 5. Minimal cut set kézzel és algoritmikusan

OR kapunál a gyermekek cut setjeit egyesítjük; AND kapunál a gyermekek cut setjeinek kombinációit képezzük, majd eltávolítjuk a nem minimális szuperszetteket.


In [ ]:
from itertools import product
def cut_sets(tree,node_id=None):
    node_id=node_id or tree["top_event"]; node=tree["nodes"][node_id]
    if node["type"]=="BASIC": return [frozenset([node_id])]
    groups=[cut_sets(tree,c) for c in node["children"]]
    cand=[s for g in groups for s in g] if node["type"]=="OR" else [frozenset().union(*x) for x in product(*groups)]
    uniq=sorted(set(cand),key=lambda s:(len(s),sorted(s))); out=[]
    for s in uniq:
        if not any(x<=s for x in out): out.append(s)
    return out
for s in cut_sets(tk): print(sorted(s))


## 6. Birnbaum importance

Oktatási érzékenységi mérő: az adott basic eventet egyszer biztosan hibásnak (`p=1`), egyszer biztosan működőnek (`p=0`) állítjuk, és vesszük a top-event probability különbségét.


In [ ]:
from copy import deepcopy
def birnbaum(tree):
    out={}
    for e,n in tree["nodes"].items():
        if n["type"]!="BASIC": continue
        a,b=deepcopy(tree),deepcopy(tree)
        a["nodes"][e]["probability"]=1.; b["nodes"][e]["probability"]=0.
        out[e]=evaluate(a)-evaluate(b)
    return pd.Series(out).sort_values(ascending=False)
imp=birnbaum(tk); display(imp)
imp.plot(kind="barh"); plt.xlabel("Birnbaum importance"); plt.show()


## 7. Ellenőrzés a tesztelt `safetycourse.fta` modullal

Most, hogy az algoritmust felépítettük, a közös modul az automatizálást és a későbbi integrációt szolgálja.


In [ ]:
from safetycourse.fta import top_event_probability,minimal_cut_sets
print(top_event_probability(tk))
print([sorted(x) for x in minimal_cut_sets(tk)])


## 8. Korlátok

A statikus FTA nem kezeli természetesen az eseménysorrendet és az időt. A gate-inputok függetlensége nem általános: shared basic event és common-cause failure külön modellezést igényel.


## Próbáld ki!

1. Növeld LT101_LOW értékét 0.10-re és számítsd újra a top eventet.
2. Mely basic event Birnbaum importance-a a legnagyobb és miért?
3. Adj COMMON_POWER_FAIL eseményt, amely egyszerre érintheti LSHH és XV funkciót; miért nem helyes két független ágra másolni?
4. Rajzold fel a négy minimal cut setet kézzel.


## Összefoglalás

Az FTA formalizálja a kvalitatív veszélyelemzésből származó kombinációs logikát. A következő tanegység azt vizsgálja, hogyan származtatható ez a struktúra **rendszermodellből**, determinisztikus és validálható szabályokkal.
